In [2]:
import os
print(os.listdir("/kaggle/input/datasets/kunjagarwal/mt-en-hi-project"))

['src', 'data']


In [3]:
import sys
sys.path.append("/kaggle/input/datasets/kunjagarwal/mt-en-hi-project")

In [6]:
import shutil
import os

src_path = "/kaggle/input/datasets/kunjagarwal/mt-en-hi-project/data/processed/iitb_filtered"  # confirm this path is correct first
dst_path = "/kaggle/working/iitb_filtered"

shutil.copytree(src_path, dst_path)

from datasets import load_from_disk
dataset_filtered = load_from_disk(dst_path)
print(dataset_filtered)

DatasetDict({
    train: Dataset({
        features: ['en', 'hi'],
        num_rows: 1413872
    })
    validation: Dataset({
        features: ['en', 'hi'],
        num_rows: 520
    })
    test: Dataset({
        features: ['en', 'hi'],
        num_rows: 2505
    })
})


In [7]:
import pickle
from datasets import load_from_disk
sys.path.append("/kaggle/input/datasets/kunjagarwal/mt-en-hi-project/src")

with open('/kaggle/input/datasets/kunjagarwal/mt-en-hi-project/data/processed/en_vocab.pkl', 'rb') as f:
    en_vocab = pickle.load(f)
with open('/kaggle/input/datasets/kunjagarwal/mt-en-hi-project/data/processed/hi_vocab.pkl', 'rb') as f:
    hi_vocab = pickle.load(f)

SUBSET_SIZE = 180_000
train_subset = dataset_filtered["train"].shuffle(seed=42).select(range(SUBSET_SIZE))
val_subset = dataset_filtered["validation"].shuffle(seed=42).select(range(min(10_000, len(dataset_filtered["validation"]))))
test_subset = dataset_filtered["test"].shuffle(seed=42).select(range(min(10_000, len(dataset_filtered["test"]))))

In [11]:
from dataset import TranslationDataset, collate_fn
from torch.utils.data import DataLoader
from functools import partial

train_en = [ex["en"] for ex in train_subset]
train_hi = [ex["hi"] for ex in train_subset]
val_en = [ex["en"] for ex in val_subset]
val_hi = [ex["hi"] for ex in val_subset]

train_dataset = TranslationDataset(train_en, train_hi, en_vocab, hi_vocab)
val_dataset = TranslationDataset(val_en, val_hi, en_vocab, hi_vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                           collate_fn=partial(collate_fn, pad_idx=en_vocab.PAD_IDX))
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,
                         collate_fn=partial(collate_fn, pad_idx=en_vocab.PAD_IDX))

In [12]:
import torch
from models.gru_attention import Encoder, Decoder, Attention, Seq2Seq

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

EMBED_DIM = 128
HIDDEN_DIM = 256

encoder = Encoder(len(en_vocab), EMBED_DIM, HIDDEN_DIM, pad_idx=en_vocab.PAD_IDX).to(device)
decoder = Decoder(len(hi_vocab), EMBED_DIM, HIDDEN_DIM, pad_idx=hi_vocab.PAD_IDX).to(device)
model = Seq2Seq(encoder, decoder, device, start_idx=hi_vocab.START_IDX,
                end_idx=hi_vocab.END_IDX, pad_idx=en_vocab.PAD_IDX).to(device)

criterion = torch.nn.CrossEntropyLoss(ignore_index=hi_vocab.PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

cuda


In [14]:
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

In [15]:
from train_attention import train_epoch, evaluate_epoch

for epoch in range(2):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, teacher_forcing_ratio=0.5)
    val_loss = evaluate_epoch(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/3 — Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}")

    torch.save(model.state_dict(), f'/kaggle/working/gru_attention_epoch{epoch+1}.pt')

  Batch 0/5625 — loss: 7.7754
  Batch 50/5625 — loss: 7.5939
  Batch 100/5625 — loss: 7.4955
  Batch 150/5625 — loss: 7.4050
  Batch 200/5625 — loss: 6.9939
  Batch 250/5625 — loss: 7.4839
  Batch 300/5625 — loss: 7.2641
  Batch 350/5625 — loss: 7.1658
  Batch 400/5625 — loss: 6.8722
  Batch 450/5625 — loss: 7.0865
  Batch 500/5625 — loss: 7.0639
  Batch 550/5625 — loss: 6.4898
  Batch 600/5625 — loss: 6.7938
  Batch 650/5625 — loss: 6.9379
  Batch 700/5625 — loss: 6.7052
  Batch 750/5625 — loss: 6.6380
  Batch 800/5625 — loss: 6.5751
  Batch 850/5625 — loss: 6.6564
  Batch 900/5625 — loss: 6.7802
  Batch 950/5625 — loss: 6.6200
  Batch 1000/5625 — loss: 6.6982
  Batch 1050/5625 — loss: 6.6782
  Batch 1100/5625 — loss: 6.5893
  Batch 1150/5625 — loss: 6.9245
  Batch 1200/5625 — loss: 6.6347
  Batch 1250/5625 — loss: 6.7124
  Batch 1300/5625 — loss: 6.5585
  Batch 1350/5625 — loss: 6.5139
  Batch 1400/5625 — loss: 6.7557
  Batch 1450/5625 — loss: 6.4823
  Batch 1500/5625 — loss: 6.6200


In [17]:
from train_attention import translate_sentence

eval_model = model.module if isinstance(model, torch.nn.DataParallel) else model

predictions = []
references = []

for ex in test_subset:
    pred = translate_sentence(eval_model, ex["en"], en_vocab, hi_vocab, device)
    predictions.append(pred)
    references.append(ex["hi"])

In [18]:
import sacrebleu
bleu = sacrebleu.corpus_bleu(predictions, [references])
print(f"BLEU score: {bleu.score:.2f}")

BLEU score: 0.61


In [19]:
print(test_subset[0]["en"])
print(test_subset[0]["hi"])

They never believed he died the way the sheriff concluded.
उन्हें शेरिफ के बताये गये तरीके से मृत्यु पर बिलकुल भी भरोसा नहीं है।


In [20]:
print(predictions[0])
print(references[0])

कभी कभी नहीं नहीं नहीं
उन्हें शेरिफ के बताये गये तरीके से मृत्यु पर बिलकुल भी भरोसा नहीं है।


In [22]:
import json

gru_results = {
    "model": "GRU with attention",
    "epochs_trained": 2,
    "embed_dim": 128,
    "hidden_dim": 256,
    "subset_size": 180_000,
    "bleu_score": bleu.score,
}

with open("gru_attention_results.json", "w") as f:
    json.dump(gru_results, f, indent=2, ensure_ascii=False)

In [23]:
torch.save(model.state_dict(), "model.pt")